# Table 2: Few-Shot Learning 实验

本 Notebook 实现论文 Table 2 的 few-shot 学习实验，包含 **VDLF-Net** 与基线方法（Prototypical、Matching、MAML）的对比。

---

## 实验运行流程（请按顺序执行）

| 步骤 | 操作 | 说明 |
|------|------|------|
| 1 | **修改并运行「0. 全局参数」** | 设置 N,K,Q、episodes、SKIP_MAML 等，快速测试可设 `NUM_TRAIN_EPISODES=50`，`NUM_TEST_EPISODES=10` |
| 2 | **运行 Section 1–2** | 环境导入、 MiniImageNet 数据加载 |
| 3 | **运行 Section 3** | 加载数据并创建采样器（根据 K_SHOT 变化） |
| 4 | **运行 Section 4** | VDLF-Net 配置与训练（或加载 checkpoint 评估） |
| 5 | **运行 Section 5** | 基线方法：Proto、Match、MAML（可跳过） |
| 6 | **运行「保存当前 K_SHOT 结果」** | 根据 K_SHOT 自动写入 `results_1shot`（K=1）或 `results_5shot`（K=5） |
| 7 | **运行「Table 2 汇总」** | 打印 ACC、CI、训练/测试耗时的汇总表 |

**完整 Table 2**：先设 `K_SHOT=1` 跑完 1–6，再设 `K_SHOT=5` 重跑 3–6，最后运行 Section 7 得到完整表格。

## 0. 全局实验参数

**修改下方参数后，后续所有实验将使用新配置。** 师弟调参时只需改这里。

In [1]:
# ========== 0. 全局实验参数（修改此处即可影响后续所有实验）==========
# Episode 设置
N_WAY = 5
K_SHOT = 5          # 1 或 5，对应 1-shot / 5-shot 实验
Q_QUERY = 15

# 训练与测试 episode 数量
# 50 ep：VDLF 之前在此设置下曾达 ~64%（若用旧 ckpt）；200 ep 时 Val 卡在 ~40%，疑似过拟合/学习率问题
NUM_TRAIN_EPISODES = 150    # 多训一点，配合更密验证捕捉最佳 ckpt
NUM_TEST_EPISODES = 100     # 10=快速；100=更稳定估计；600=正式

# 基线 MAML：True=跳过（需 learn2learn，Windows 可能需 Visual C++ 编译）
SKIP_MAML = True

# VDLF-Net：True=跳过训练，直接加载 best_vdlf_model.pth 评估（节省时间）
LOAD_VDLF_CKPT = False   # 有现成模型时 True；首次训练或重训时 False

# 其他
IMAGE_SIZE = 84
BATCH_SIZE = 1     # 显存紧，配合梯度累积模拟 batch=4
GRADIENT_ACCUMULATION_STEPS = 4  # 等效 batch=4，梯度更稳
LEARNING_RATE = 1e-4    # 与 Proto/Match 对齐；0.0008 偏高易震荡
WEIGHT_DECAY = 0.00005

# VDLF-Net 变分相关（论文中的 T、τ、α）
NUM_SAMPLES = 15   # T：潜在采样数；梯度累积后每次只处理 1 ep，可略大
TEMPERATURE = 15.0 # τ：cosine-softmax 温度，越大决策越锐
ALPHA = 0.01       # α：降低变分正则，更偏向判别损失

# 打印当前配置
print("当前实验配置:")
print(f"  N-way / K-shot / Q-query: {N_WAY} / {K_SHOT} / {Q_QUERY}")
print(f"  train_episodes: {NUM_TRAIN_EPISODES}, test_episodes: {NUM_TEST_EPISODES}")
print(f"  VDLF: T={NUM_SAMPLES}, τ={TEMPERATURE}, α={ALPHA}, batch×accum={BATCH_SIZE}×{GRADIENT_ACCUMULATION_STEPS}")
print(f"  SKIP_MAML: {SKIP_MAML}, LOAD_VDLF_CKPT: {LOAD_VDLF_CKPT}")

当前实验配置:
  N-way / K-shot / Q-query: 5 / 5 / 15
  train_episodes: 150, test_episodes: 100
  VDLF: T=15, τ=15.0, α=0.01, batch×accum=1×4
  SKIP_MAML: True, LOAD_VDLF_CKPT: False


## 1. 环境与导入

In [2]:
# 导入依赖
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))  # 确保能导入 experiment_framework
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

# 导入实验框架
from experiment_framework import FewShotConfig, EpisodeSampler

# 数据路径（相对于项目根目录）
DATA_ROOT = Path("data/mini-imagenet")
print(f"数据路径: {DATA_ROOT.absolute()}")
print(f"数据集存在: {DATA_ROOT.exists()}")
print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {'cuda' if torch.cuda.is_available() else 'cpu'}")

数据路径: c:\Users\15366\Desktop\experiment_2\data\mini-imagenet
数据集存在: True
PyTorch 版本: 2.9.1+cu130
设备: cuda


d:\Anaconda\envs\Paper3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Mini-ImageNet 数据集与变换

In [3]:
import pickle
from collections import defaultdict

class MiniImageNetDataset(Dataset):
    """Mini-ImageNet 数据集，支持索引缓存以加速加载"""
    
    def __init__(self, root: Path, split: str = "train", transform=None, cache_dir: Path = None):
        """
        Args:
            root: 数据集根目录 (包含 train/val/test 子目录)
            split: 'train', 'val', 或 'test'
            transform: 图像变换
            cache_dir: 索引缓存目录，None 则用 root/.cache
        """
        self.root = Path(root)
        self.split = split
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        cache_path = (cache_dir or self.root / ".cache") / f"{split}_index.pkl"
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        
        # 尝试从缓存加载
        if cache_path.exists():
            with open(cache_path, "rb") as f:
                cached = pickle.load(f)
            self.images = cached["images"]
            self.labels = cached["labels"]
            self.class_to_idx = cached["class_to_idx"]
            print(f"{split}: {len(self.images)} 张图像, {len(self.class_to_idx)} 个类别 (从缓存加载)")
            return
        
        split_dir = self.root / split
        if not split_dir.exists():
            raise FileNotFoundError(f"目录不存在: {split_dir}")
        
        # 遍历类别目录（使用 os.listdir 加速，比 pathlib.glob 快）
        import os
        for class_name in sorted(os.listdir(split_dir)):
            class_dir = split_dir / class_name
            if class_dir.is_dir():
                if class_name not in self.class_to_idx:
                    self.class_to_idx[class_name] = len(self.class_to_idx)
                class_idx = self.class_to_idx[class_name]
                for f in os.listdir(class_dir):
                    if f.lower().endswith(".jpg"):
                        self.images.append(str(class_dir / f))
                        self.labels.append(class_idx)
        
        # 保存缓存
        with open(cache_path, "wb") as f:
            pickle.dump({"images": self.images, "labels": self.labels, "class_to_idx": self.class_to_idx}, f, protocol=4)
        print(f"{split}: {len(self.images)} 张图像, {len(self.class_to_idx)} 个类别 (已缓存)")
    
    def get_class_to_indices(self):
        """返回 {label: [indices]}，无需加载图像，供 EpisodeSampler 使用"""
        d = defaultdict(list)
        for idx, label in enumerate(self.labels):
            d[label].append(idx)
        return dict(d)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def get_transforms(split: str = "train", image_size: int = 84):
    """获取数据增强变换"""
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    
    if split == "train":
        return transforms.Compose([
            transforms.Resize(int(image_size * 1.1)),
            transforms.RandomCrop(image_size, padding=8),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ToTensor(),
            normalize
        ])
    else:
        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            normalize
        ])

**加速说明**：索引会缓存到 `data/mini-imagenet/.cache/`，首次运行需遍历磁盘（约 1–3 分钟），之后从缓存加载，几秒内完成。若更换或更新数据集，请删除 `.cache` 目录后重新运行。

## 3. 加载数据集并创建 Episode 采样器

**说明**：`N_WAY`、`K_SHOT`、`Q_QUERY` 来自 Section 0。修改 `K_SHOT` 后需重新运行本 cell 以更新采样器。

In [4]:
# 加载 train / val / test（首次运行需遍历磁盘，之后从缓存加载）
# N_WAY, K_SHOT, Q_QUERY, IMAGE_SIZE 来自 Section 0
train_dataset = MiniImageNetDataset(
    DATA_ROOT, split="train",
    transform=get_transforms("train", IMAGE_SIZE)
)
val_dataset = MiniImageNetDataset(
    DATA_ROOT, split="val",
    transform=get_transforms("val", IMAGE_SIZE)
)
test_dataset = MiniImageNetDataset(
    DATA_ROOT, split="test",
    transform=get_transforms("val", IMAGE_SIZE)
)

# 创建 Episode 采样器
train_sampler = EpisodeSampler(train_dataset, N_WAY, K_SHOT, Q_QUERY, 
                               class_to_indices=train_dataset.get_class_to_indices())
val_sampler = EpisodeSampler(val_dataset, N_WAY, K_SHOT, Q_QUERY,
                             class_to_indices=val_dataset.get_class_to_indices())
test_sampler = EpisodeSampler(test_dataset, N_WAY, K_SHOT, Q_QUERY,
                             class_to_indices=test_dataset.get_class_to_indices())

print(f"\nEpisode 配置: {N_WAY}-way {K_SHOT}-shot, {Q_QUERY} query/class")

train: 38400 张图像, 64 个类别 (从缓存加载)
val: 9600 张图像, 16 个类别 (从缓存加载)
test: 12000 张图像, 20 个类别 (从缓存加载)

Episode 配置: 5-way 5-shot, 15 query/class


### 3.1 验证集与测试集的使用

| 数据集 | 对应变量 | 使用位置 | 用途 |
|--------|----------|----------|------|
| **训练集** | `train_sampler` | `train_vdlf_net(..., train_sampler, ...)` | 训练时采样 episode、计算损失、梯度更新 |
| **验证集** | `val_sampler` | `train_vdlf_net(..., val_sampler, ...)` 内的 `evaluate_model(model, val_sampler, ...)` | 每 `val_interval` 个 episode 评估一次，用于保存最佳模型 |
| **测试集** | `test_sampler` | 训练完成后，「加载最佳模型并在测试集上评估」cell | 最终报告准确率，**训练过程从未使用** |

### 3.1 验证数据加载

In [5]:
# 采样一个 episode 并检查形状
support_imgs, support_labels, query_imgs, query_labels = train_sampler.sample_episode()

print("Support set:")
print(f"  - images: {support_imgs.shape}")
print(f"  - labels: {support_labels.shape}")
print("\nQuery set:")
print(f"  - images: {query_imgs.shape}")
print(f"  - labels: {query_labels.shape}")
print("\n[PASS] 数据加载正常，可以开始训练。")

Support set:
  - images: torch.Size([25, 3, 84, 84])
  - labels: torch.Size([25])

Query set:
  - images: torch.Size([75, 3, 84, 84])
  - labels: torch.Size([75])

[PASS] 数据加载正常，可以开始训练。


## 4. VDLF-Net 训练与评估

**LOAD_VDLF_CKPT=True**：跳过训练，直接加载 `best_vdlf_model.pth` 评估（训练耗时为 —）
**LOAD_VDLF_CKPT=False**：完整训练，进度和耗时会在下方显示。

In [6]:
# 导入 VDLF-Net 及相关函数
from experiment_framework import (
    FewShotConfig, VDLFNet, compute_vdlf_loss,
    evaluate_model, train_vdlf_net
)

# 配置（参数来自 Section 0）
config = FewShotConfig(
    n_way=N_WAY,
    k_shot=K_SHOT,
    q_query=Q_QUERY,
    num_train_episodes=NUM_TRAIN_EPISODES,
    num_test_episodes=NUM_TEST_EPISODES,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    latent_dim=128,
    num_samples=NUM_SAMPLES,
    temperature=TEMPERATURE,
    alpha=ALPHA,
)

# 创建模型
model = VDLFNet(config)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

模型参数量: 42,757,122


### 4.1 训练与评估（二选一或按序执行）

- **训练**：`LOAD_VDLF_CKPT=False` 时运行，会显示进度条和验证准确率，训练结束后打印耗时统计
- **评估**：始终运行，加载最佳 checkpoint 在测试集上评估，打印 ACC ± CI 和测试耗时

In [7]:
# 训练（LOAD_VDLF_CKPT=False 时执行）
if LOAD_VDLF_CKPT:
    vdlf_train_time = None
    print("已跳过 VDLF 训练（LOAD_VDLF_CKPT=True）")
else:
    print("开始训练 VDLF-Net...")
    model, vdlf_train_time = train_vdlf_net(model, train_sampler, val_sampler, config)
    print("训练完成。")

# 加载最佳模型并在测试集上评估（始终执行）
import os
ckpt_path = "best_vdlf_model.pth"
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=config.device))
else:
    print("未找到 checkpoint，使用当前模型权重评估。")
model = model.to(config.device)

mean_acc, ci_95, accuracies, vdlf_test_time = evaluate_model(model, test_sampler, config)

print(f"\n{'='*50}")
print(f"VDLF-Net 测试集结果 ({config.num_test_episodes} episodes)")
print(f"{'='*50}")
print(f"5-way {K_SHOT}-shot 准确率: {mean_acc*100:.2f}% ± {ci_95*100:.2f}%")
print(f"测试耗时: {vdlf_test_time:.1f}s ({vdlf_test_time/config.num_test_episodes:.2f}s/ep)")
print(f"{'='*50}")

开始训练 VDLF-Net...
进入训练循环，首个 batch 可能较慢（GPU 预热）...


Training:   0%|          | 0/150 [00:00<?, ?ep/s]

首个 batch 完成 (耗时 2.1s)，进度条已启动


Training:  17%|█▋        | 25/150 [00:52<10:50,  5.21s/ep]

Ep 25/150 | Val: 83.13% ± 1.16% (saved)


...

Training: 100%|██████████| 150/150 [05:06<00:00,  2.04s/ep]

Ep 150/150 | Val: 85.80% ± 1.26% (saved)

--- 训练时间统计 ---
总耗时:        306.4s (5.1 min)
训练耗时:      236.0s (不含验证)
验证耗时:      70.4s (共 6 次)
首次 batch:    2.1s (GPU 预热)
平均每 ep:     1.57s
-------------------


训练完成。


Evaluating: 100%|██████████| 100/100 [00:13<00:00,  7.67ep/s]


VDLF-Net 测试集结果 (100 episodes)
5-way 5-shot 准确率: 86.33% ± 1.17%
测试耗时: 13.0s (0.13s/ep)


## 5. 基线方法（Proto / Match / MAML）

在相同 episode 参数下训练并评估，与 VDLF-Net 对比。Proto/Match 用 lr=1e-4；MAML 可通过 Section 0 的 `SKIP_MAML` 跳过。

In [8]:
# Prototypical Networks
from baselines import PrototypicalNetworks, train_prototypical, evaluate_baseline

proto_model = PrototypicalNetworks()
print("训练 Prototypical Networks...")
proto_model, proto_train_time = train_prototypical(proto_model, train_sampler, val_sampler, config)

ckpt = "best_protonet.pth"
if os.path.exists(ckpt):
    proto_model.load_state_dict(torch.load(ckpt, map_location=config.device))
proto_model = proto_model.to(config.device)
proto_acc, proto_ci, proto_test_time = evaluate_baseline(proto_model, test_sampler, config, N_WAY, K_SHOT, "protonet")
print(f"Prototypical Networks: {proto_acc*100:.2f}% ± {proto_ci*100:.2f}%")
print(f"训练耗时: {proto_train_time:.1f}s | 测试耗时: {proto_test_time:.1f}s ({proto_test_time/config.num_test_episodes:.2f}s/ep)")

训练 Prototypical Networks...


ProtoNet:   7%|▋         | 11/150 [00:05<00:26,  5.23ep/s]

Ep 15 | Val: 77.40%


ProtoNet:  19%|█▉        | 29/150 [00:10<00:31,  3.87ep/s]

Ep 30 | Val: 69.27%


...

Ep 135 | Val: 62.60%


ProtoNet: 100%|██████████| 150/150 [00:49<00:00,  3.02ep/s]


Ep 150 | Val: 57.07%


Eval protonet: 100%|██████████| 100/100 [00:12<00:00,  8.33ep/s]

Prototypical Networks: 80.09% ± 1.14%
训练耗时: 49.7s | 测试耗时: 12.0s (0.12s/ep)


In [9]:
# Matching Networks
from baselines import MatchingNetworks, train_matching, evaluate_baseline

match_model = MatchingNetworks()
print("训练 Matching Networks...")
match_model, match_train_time = train_matching(match_model, train_sampler, val_sampler, config)

ckpt = "best_matchnet.pth"
if os.path.exists(ckpt):
    match_model.load_state_dict(torch.load(ckpt, map_location=config.device))
match_model = match_model.to(config.device)
match_acc, match_ci, match_test_time = evaluate_baseline(match_model, test_sampler, config, N_WAY, K_SHOT, "matchnet")
print(f"Matching Networks: {match_acc*100:.2f}% ± {match_ci*100:.2f}%")
print(f"训练耗时: {match_train_time:.1f}s | 测试耗时: {match_test_time:.1f}s ({match_test_time/config.num_test_episodes:.2f}s/ep)")

训练 Matching Networks...


MatchNet:   8%|▊         | 12/150 [00:05<00:26,  5.26ep/s]

Ep 15 | Val: 73.07%


MatchNet:  20%|██        | 30/150 [00:10<00:49,  2.41ep/s]

Ep 30 | Val: 72.73%


...

Ep 135 | Val: 69.00%


MatchNet: 100%|██████████| 150/150 [00:49<00:00,  3.02ep/s]


Ep 150 | Val: 61.47%


Eval matchnet: 100%|██████████| 100/100 [00:11<00:00,  8.62ep/s]

Matching Networks: 68.87% ± 1.87%
训练耗时: 49.6s | 测试耗时: 11.6s (0.12s/ep)


In [10]:
# MAML（SKIP_MAML 在 Section 0 中设置，True=跳过；需 pip install learn2learn）
from baselines import get_maml_model, train_maml, evaluate_maml, L2L_AVAILABLE

if not SKIP_MAML and L2L_AVAILABLE:
    maml_model = get_maml_model(N_WAY)
    print("训练 MAML...")
    maml_model, maml_train_time = train_maml(maml_model, train_sampler, val_sampler, config, adaptation_steps=5, meta_batch_size=4)

    ckpt = "best_maml.pth"
    if os.path.exists(ckpt):
        maml_model.load_state_dict(torch.load(ckpt, map_location=config.device))
    maml_model = maml_model.to(config.device)
    maml_acc, maml_ci, maml_test_time = evaluate_maml(maml_model, test_sampler, config, N_WAY, K_SHOT, adaptation_steps=5)
    print(f"MAML: {maml_acc*100:.2f}% ± {maml_ci*100:.2f}%")
    print(f"训练耗时: {maml_train_time:.1f}s | 测试耗时: {maml_test_time:.1f}s ({maml_test_time/config.num_test_episodes:.2f}s/ep)")
else:
    if SKIP_MAML:
        print("已跳过 MAML（SKIP_MAML=True）")
    else:
        print("未安装 learn2learn，跳过 MAML。运行: pip install learn2learn")
    maml_acc, maml_ci, maml_train_time, maml_test_time = None, None, None, None

已跳过 MAML（SKIP_MAML=True）


In [11]:
# 当前 K_SHOT 结果表（即时打印 ACC、CI、训练/测试耗时）
def _acc_str(a, c):
    return f"{a*100:.2f}% ± {c*100:.2f}%" if (a is not None and c is not None) else "—"
def _t_str(t):
    return f"{t:.1f}s" if t is not None else "—"

# 安全获取变量（某模型未运行时为 None）
def _get(name):
    try: return eval(name)
    except NameError: return None

rows = [
    ("MAML", _get("maml_acc"), _get("maml_ci"), _get("maml_train_time"), _get("maml_test_time")),
    ("Prototypical Networks", _get("proto_acc"), _get("proto_ci"), _get("proto_train_time"), _get("proto_test_time")),
    ("Matching Networks", _get("match_acc"), _get("match_ci"), _get("match_train_time"), _get("match_test_time")),
    ("VDLF-Net", _get("mean_acc"), _get("ci_95"), _get("vdlf_train_time"), _get("vdlf_test_time")),
]

print(f"\n{'='*70}")
print(f"5-way {K_SHOT}-shot 结果 | train_ep={config.num_train_episodes}, test_ep={config.num_test_episodes}")
print(f"{'='*70}")
print(f"| {'Model':<25} | {'Acc ± CI':<20} | {'训练耗时':<12} | {'测试耗时':<12} |")
print(f"|{'-'*27}|{'-'*22}|{'-'*14}|{'-'*14}|")
for name, acc, ci, tr, te in rows:
    _tr = _t_str(tr) if tr is not None else "—"
    _te = _t_str(te) if te is not None else "—"
    print(f"| {name:<25} | {_acc_str(acc, ci):<20} | {_tr:<12} | {_te:<12} |")
print(f"{'='*70}")


5-way 5-shot 结果 | train_ep=150, test_ep=100
| Model                     | Acc ± CI             | 训练耗时         | 测试耗时         |
|---------------------------|----------------------|--------------|--------------|
| MAML                      | —                    | —            | —            |
| Prototypical Networks     | 80.09% ± 1.14%       | 49.7s        | 12.0s        |
| Matching Networks         | 68.87% ± 1.87%       | 49.6s        | 11.6s        |
| VDLF-Net                  | 86.33% ± 1.17%       | 306.4s       | 13.0s        |


## 6. 保存结果与 Table 2 汇总

1. **保存当前 K_SHOT 结果**：运行下方 cell，会根据当前 K_SHOT 自动存入 `results_1shot`（K=1）或 `results_5shot`（K=5）
2. **完整 Table 2**：设 `K_SHOT=1` 跑完 Section 3–5，运行保存 cell；再设 `K_SHOT=5` 重跑 Section 3–5，再运行保存 cell；最后运行「Table 2 汇总」
3. **持久化**（可选）：运行「持久化到文件」将结果写入 `experiment_log.json` 和 `EXPERIMENT_RESULTS.md`

In [12]:
# 保存当前 K_SHOT 结果（根据 Section 0 的 K_SHOT 自动保存到 results_1shot 或 results_5shot）
try:
    maml_tup = (maml_acc, maml_ci, maml_train_time, maml_test_time)
except NameError:
    maml_tup = (None, None, None, None)

def _get_vdlf():
    try: acc = mean_acc
    except NameError: acc = None
    try: ci = ci_95
    except NameError: ci = None
    try: tr = vdlf_train_time
    except NameError: tr = None
    try: te = vdlf_test_time
    except NameError: te = None
    return (acc, ci, tr, te)

row = {
    'MAML': maml_tup,
    'Prototypical Networks': (proto_acc, proto_ci, proto_train_time, proto_test_time),
    'Matching Networks': (match_acc, match_ci, match_train_time, match_test_time),
    'VDLF-Net': _get_vdlf(),
}

if K_SHOT == 1:
    results_1shot = row
    print("已保存到 results_1shot（当前 K_SHOT=1）")
elif K_SHOT == 5:
    results_5shot = row
    print("已保存到 results_5shot（当前 K_SHOT=5）")
else:
    print(f"[警告] K_SHOT={K_SHOT} 不是 1 或 5，未保存。请设置 K_SHOT=1 或 5 后重跑 Section 3–5 再运行本 cell。")

已保存到 results_5shot（当前 K_SHOT=5）


In [13]:
# 持久化到文件（可选）
# 将 results_1shot、results_5shot 写入 experiment_log.json 和 EXPERIMENT_RESULTS.md
from save_experiment_results import save_experiment

try:
    r1 = results_1shot
except NameError:
    r1 = {}
try:
    r5 = results_5shot
except NameError:
    r5 = {}
try:
    cfg = config
except NameError:
    cfg = None
try:
    note = f"train_ep={cfg.num_train_episodes}, test_ep={cfg.num_test_episodes}" if cfg else ""
except Exception:
    note = ""

save_experiment(cfg, r1, r5, note=note)

已保存到 c:\Users\15366\Desktop\experiment_2\experiment_log.json 和 c:\Users\15366\Desktop\experiment_2\EXPERIMENT_RESULTS.md


In [14]:
# Table 2 汇总输出：Markdown + LaTeX（含耗时）
def fmt(acc, ci):
    if acc is None or ci is None:
        return "—"
    return f"{acc*100:.2f} $\\pm$ {ci*100:.2f}"

def fmt_time(t):
    if t is None: return "—"
    return f"{t/60:.1f}min" if t >= 60 else f"{t:.0f}s"

models = ['MAML', 'Prototypical Networks', 'Matching Networks', 'VDLF-Net']
try:
    r1 = results_1shot
except NameError:
    r1 = {}
try:
    r5 = results_5shot
except NameError:
    r5 = {}

def get_row(d, name):
    tup = d.get(name, (None, None, None, None))
    if len(tup) >= 4:
        return tup[0], tup[1], tup[2], tup[3]
    return tup[0], tup[1], None, None

data = [(m, get_row(r1, m), get_row(r5, m)) for m in models]

# Markdown 表格
print("=" * 70)
print("【Markdown】复制下方内容：")
print("=" * 70)
md = "| Model | 1-shot (%) | 5-shot (%) | 训练1s | 测试1s | 训练5s | 测试5s |\n|-------|------------|------------|--------|--------|--------|--------|\n"
for m, (a1,c1,tr1,te1), (a5,c5,tr5,te5) in data:
    s1 = fmt(a1,c1).replace("$\\pm$", "±") if a1 else "—"
    s5 = fmt(a5,c5).replace("$\\pm$", "±") if a5 else "—"
    md += f"| {m} | {s1} | {s5} | {fmt_time(tr1)} | {fmt_time(te1)} | {fmt_time(tr5)} | {fmt_time(te5)} |\n"
print(md)

【Markdown】复制下方内容：
| Model | 1-shot (%) | 5-shot (%) | 训练1s | 测试1s | 训练5s | 测试5s |
|-------|------------|------------|--------|--------|--------|--------|
| MAML | — | — | — | — | — | — |
| Prototypical Networks | — | 80.09 ± 1.14 | — | — | 50s | 12s |
| Matching Networks | — | 68.87 ± 1.87 | — | — | 50s | 12s |
| VDLF-Net | — | 86.33 ± 1.17 | — | — | 5.1min | 13s |

